# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
print("hello world!")

hello world!


In [14]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
from langsmith import traceable

In [3]:
# Initialize and constants

load_dotenv(override=True)
google_api_key = os.getenv('GEMINI_API_KEY')
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

MODEL = 'gemini-3.5-flash'
# openai = OpenAI()
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

In [4]:
response = gemini.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'A typical fluffy cumulus cloud weighs about **1.1 million pounds** (around 500,000 kg)—which is roughly the same weight as **100 elephants**! \n\nEven though they look like weightless cotton candy, they are actually made of tons of tiny water droplets. They stay afloat because the air beneath them is warmer and denser than the air inside the cloud, acting like a giant, invisible cushion.'

In [5]:
links = fetch_website_links("https://www.krishnaik.in/")
links

['/',
 'https://krishcnaik.substack.com/',
 '/liveclasses',
 '/projects',
 '/ai-pro-subscription',
 '/courses',
 '/ai-roadmaps',
 '/webinars',
 '/corporate-training',
 'https://apps.apple.com/us/app/krish-naik-academy/id6742422392',
 'https://play.google.com/store/apps/details?id=com.tagmango.krishnaikacademy&pcampaignid=web_share',
 '/courses',
 '/projects',
 '/liveclasses',
 '/ai-roadmaps',
 'https://www.youtube.com/@krishnaik06',
 '/courses',
 '/courses',
 '/courses',
 'https://apps.apple.com/us/app/krish-naik-academy/id6742422392',
 'https://play.google.com/store/apps/details?id=com.tagmango.krishnaikacademy&pcampaignid=web_share',
 '/liveclasses',
 '/liveclass2/Advanded-Route?id=12',
 '/liveclass2/Datascience?id=13',
 '/liveclass2/AgenticAi?id=11',
 '/liveclasses',
 '/courses',
 '/courses',
 'https://www.udemy.com/course/building-ai-agents-agentic-ai-system-via-microsoft-autogen',
 'https://www.udemy.com/course/ai-powered-data-analytics-mastery-ai-tools-vibe-coding/',
 'https://ww

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [15]:
@traceable
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://www.krishnaik.in/"))


Here is the list of links on the website https://www.krishnaik.in/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
https://krishcnaik.substack.com/
/liveclasses
/projects
/ai-pro-subscription
/courses
/ai-roadmaps
/webinars
/corporate-training
https://apps.apple.com/us/app/krish-naik-academy/id6742422392
https://play.google.com/store/apps/details?id=com.tagmango.krishnaikacademy&pcampaignid=web_share
/courses
/projects
/liveclasses
/ai-roadmaps
https://www.youtube.com/@krishnaik06
/courses
/courses
/courses
https://apps.apple.com/us/app/krish-naik-academy/id6742422392
https://play.google.com/store/apps/details?id=com.tagmango.krishnaikacademy&pcampaignid=web_share
/liveclasses
/liveclass2/Advanded-Route?id=12
/liveclass2/Datascience?id=13
/liveclass2/AgenticAi?id=11
/liveclasses
/courses
/courses
https

In [16]:
@traceable(run_type="llm")
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [17]:
select_relevant_links("https://www.krishnaik.in/")

{'links': [{'type': 'about page', 'url': 'https://www.krishnaik.in/about'},
  {'type': 'contact page', 'url': 'https://www.krishnaik.in/contact'},
  {'type': 'corporate training page',
   'url': 'https://www.krishnaik.in/corporate-training'},
  {'type': 'courses page', 'url': 'https://www.krishnaik.in/courses'},
  {'type': 'live classes page', 'url': 'https://www.krishnaik.in/liveclasses'},
  {'type': 'projects page', 'url': 'https://www.krishnaik.in/projects'}]}

In [27]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [26]:
select_relevant_links("https://www.krishnaik.in/")

Selecting relevant links for https://www.krishnaik.in/ by calling gemini-3.5-flash
Found 5 relevant links


{'links': [{'type': 'about page', 'url': 'https://www.krishnaik.in/about'},
  {'type': 'contact page', 'url': 'https://www.krishnaik.in/contact'},
  {'type': 'corporate training page',
   'url': 'https://www.krishnaik.in/corporate-training'},
  {'type': 'courses page', 'url': 'https://www.krishnaik.in/courses'},
  {'type': 'live classes page',
   'url': 'https://www.krishnaik.in/liveclasses'}]}

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [28]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [29]:
print(fetch_page_and_all_relevant_links("https://www.krishnaik.in/"))

Selecting relevant links for https://www.krishnaik.in/ by calling gemini-3.5-flash
Found 6 relevant links
## Landing Page:

Krish Naik - Master AI & Data Science | 1M+ Students Worldwide

Krish Naik
Blogs
Courses
Explore Courses
Live Classes
Join mentor-led sessions and bootcamps
Projects
Build end-to-end industry projects
AI Pro Subscription
Access premium AI learning programs
Udemy Courses
Browse self-paced Udemy courses
Roadmaps
Webinars
Corporate Training
iOS
Android
Trusted by 1M+ Students
Master AI & Data Science with
Krish Naik
Learn from an industry expert with
13+ years
of experience. Join comprehensive courses covering Machine Learning, Deep Learning, NLP, and GenAI.
Udemy Courses
Explore Projects
Live Classes
900K+
Udemy Students
4.7★
Instructor Rating
1.3M+
YouTube Subs
1M+
Students Taught
Not sure where to start?
Follow a step-by-step roadmap — know exactly what to watch and which projects to build.
View Roadmap
Our Impact
Join a Global Community
1.3M+
YouTube Subscribers


In [41]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty, funny brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [31]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5000] # Truncate if more than 5,000 characters
    return user_prompt

In [32]:
get_brochure_user_prompt("KrishNaik", "https://www.krishnaik.in/")

Selecting relevant links for https://www.krishnaik.in/ by calling gemini-3.5-flash
Found 7 relevant links


"\nYou are looking at a company called: KrishNaik\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nKrish Naik - Master AI & Data Science | 1M+ Students Worldwide\n\nKrish Naik\nBlogs\nCourses\nExplore Courses\nLive Classes\nJoin mentor-led sessions and bootcamps\nProjects\nBuild end-to-end industry projects\nAI Pro Subscription\nAccess premium AI learning programs\nUdemy Courses\nBrowse self-paced Udemy courses\nRoadmaps\nWebinars\nCorporate Training\niOS\nAndroid\nTrusted by 1M+ Students\nMaster AI & Data Science with\nKrish Naik\nLearn from an industry expert with\n13+ years\nof experience. Join comprehensive courses covering Machine Learning, Deep Learning, NLP, and GenAI.\nUdemy Courses\nExplore Projects\nLive Classes\n900K+\nUdemy Students\n4.7★\nInstructor Rating\n1.3M+\nYouTube Subs\n1M+\nStudents Taught\nNot sure where to start?\nFollow a 

In [35]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [36]:
create_brochure("KrishNaik", "https://www.krishnaik.in/")

Selecting relevant links for https://www.krishnaik.in/ by calling gemini-3.5-flash
Found 6 relevant links


# Krish AI Technologies: Empowering the Future of AI

## Bridging the Gap Between Theory and Industry
Founded by industry expert Krish Naik, Krish AI Technologies is a premier hub for cutting-edge education and consulting in Data Science, Machine Learning, and Artificial Intelligence. With over 13 years of enterprise experience, we are dedicated to transforming learners and organizations into AI-ready leaders.

We believe that true mastery comes from practical application. Our mission is to move beyond foundational theory, helping students and enterprises alike build, deploy, and scale production-grade AI systems that solve real-world problems in sectors like finance, manufacturing, and healthcare.

---

## Why Choose Krish AI?
*   **Proven Expertise:** Learn from Krish Naik, an industry veteran with 13+ years of experience and a community of over 1.3 million YouTube subscribers.
*   **Global Impact:** Trusted by over 1 million students worldwide with a consistently high 4.8/5 average rating.
*   **Industry-Focused Curriculum:** From self-paced Udemy courses to intensive, mentor-led bootcamps, our programs cover the full spectrum of modern AI, including GenAI, NLP, MLOps, and LLM engineering.
*   **Learning On-The-Go:** Access premium content, roadmaps, and project-based learning through our dedicated iOS and Android mobile applications.

---

## Solutions for Every Need
*   **For Professionals:** Join our advanced programs, such as our 8-month production-grade LLM Engineering course, designed to help you master the architecture of high-scale AI systems.
*   **For Enterprises:** Build an AI-ready workforce with our bespoke corporate training solutions. We partner with organizations to deliver "no-fluff" L&D programs that bridge the gap from the boardroom to the engineering floor.
*   **For Aspiring Developers:** Leverage our step-by-step roadmaps and end-to-end industry projects to gain the hands-on experience required to land your next role in the AI space.

---

## Join Our Community
At Krish AI Technologies, we foster a collaborative, fast-paced culture of continuous innovation. We are passionate about upskilling professionals and driving the adoption of intelligent technologies across the globe.

**Are you ready to stay ahead in the rapidly evolving world of AI?**

*   **Explore Courses:** Visit our website to browse our full curriculum.
*   **Partner With Us:** Reach out for corporate training consultations and receive a custom proposal within 48 hours.
*   **Connect:** Follow us on YouTube and LinkedIn to stay updated on the latest industry trends, exclusive learning resources, and upcoming webinars.

**Contact Us**
Email: contact@krishnaik.in
WhatsApp Support: +91 9111533440
Web: [krishnaik.in](https://krishnaik.in)

*Krish AI Technologies — Building the AI-ready workforce of tomorrow.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [39]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [40]:
stream_brochure("AndrejKarpathy", "https://karpathy.ai/")

Selecting relevant links for https://karpathy.ai/ by calling gemini-3.5-flash
Found 7 relevant links


# Andrej Karpathy: Pushing the Frontiers of Deep Learning

### Who We Are
Andrej Karpathy is a world-renowned AI researcher, educator, and innovator dedicated to the advancement of deep neural networks. With a career spanning foundational roles at OpenAI and leadership at Tesla, Karpathy bridges the gap between complex machine learning theory and real-world, large-scale implementation. Beyond technical contributions, the organization is a hub for AI literacy, committed to demystifying the architecture of intelligence for the next generation of engineers.

### Core Philosophy: "Zero to Hero"
Our mission is built on the belief that deep learning should be accessible, transparent, and rigorous. Whether it is training neural nets on massive datasets or crafting 200-line, dependency-free Python implementations of GPT, we focus on understanding the fundamental building blocks of AI. We treat engineering as a craft—prioritizing first principles, clarity, and hands-on experimentation.

### Expertise & Impact
Our track record demonstrates a unique capability to navigate the most challenging problems in computer vision and generative AI:
*   **Autonomous Systems:** Led the computer vision team for Tesla Autopilot and Optimus, managing the end-to-end stack from data labeling to neural network deployment on custom silicon.
*   **Generative AI:** Founding member of OpenAI, with recent work focusing on midtraining and synthetic data generation.
*   **Educational Excellence:** A primary source for AI education, providing technical deep-dives into Large Language Models (LLMs) and practical guides for building robust, scalable AI systems.

### Engaging with Our Community
We believe in open knowledge. Our work is shared across multiple platforms to empower a global community of developers and learners:
*   **Educational Content:** Through the "Zero to Hero" video series and long-form technical explorations, we help practitioners gain the skills required to navigate the rapidly evolving AI landscape.
*   **Public Discourse:** We are active participants in the open-source community, sharing code, research insights, and perspectives on the future of neural network development via GitHub and social platforms.
*   **The Pursuit of Knowledge:** From AI ethics and short stories to "biohacking lite," our curiosity extends beyond code, fostering a culture of constant, multi-disciplinary learning.

### Join the Journey
Working with or learning from Andrej Karpathy means engaging with a relentless pursuit of innovation. Whether you are looking to master the technical intricacies of LLMs, understand the history and future trajectory of deep learning, or simply stay informed on the latest breakthroughs in the field, you have found the right place.

Follow our latest insights on **GitHub** and **X/Twitter**, or dive into our **YouTube channel** to begin your journey from zero to hero in the world of artificial intelligence.

In [42]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash
Found 5 relevant links


# Welcome to Hugging Face: Because Machines Need Feelings (And You Need Models)

Tired of your computer just staring at you blankly? Wish your spreadsheet had a personality, or your chatbot didn’t sound like a dial-up modem having a mid-life crisis? Welcome to **Hugging Face**, the global epicenter of AI where we help machines get a little more "human," one parameter at a time.

---

### What Exactly Is This Place?
Think of us as the **GitHub of AI**, but with more emojis and less existential dread. We are the home base for the machine learning community. Whether you are building an app that talks to you, a robot that paints, or a system that just helps you find your lost keys, we’ve got the infrastructure to make it happen.

* **2 Million+ Models:** From "slightly helpful" to "scarily brilliant."
* **500k+ Datasets:** Massive collections of data so your AI has something to read besides your embarrassing search history.
* **1 Million+ Apps:** Real-time demos where you can watch AI generate videos, talk to avatars, or edit your photos until you look like a Greek god.

---

### Why Join the Chaos?
Are you a developer, a data scientist, or just someone who thinks "AI" sounds cool at dinner parties? We want you. 

**Our Culture:** We are the collaborative heartbeat of the AI revolution. We don't just "do" machine learning; we foster an environment where geeks, geniuses, and tinkerers gather to push the boundaries of what’s possible. If you like open-source, fast-paced innovation, and the occasional cryptic math symbol as a stylistic choice, you’ll fit right in.

**Careers:** We are hiring the best and brightest to help us build the future. If you enjoy solving problems that didn’t exist three minutes ago, check our jobs page. We offer competitive benefits, the opportunity to work on bleeding-edge tech, and the chance to say, "I helped build the thing that might (or might not) take over the world."

---

### For the "Enterprise" Crowd (The Serious People)
We know, we know—your boss wants to see "ROI" and "Enterprise-Grade Scalability." Don’t worry; we speak Corporate, too. 

Our **Enterprise Solutions** provide the heavy lifting your company needs:
* **Inference Endpoints:** Because your model shouldn't just run on your laptop; it should run everywhere.
* **Storage Buckets & Pro Plans:** Secure, scalable, and—dare we say it—professional.
* **Support:** When things go wrong, we are here to hold your hand (digitally, of course).

---

### How to Get Started
1. **Explore:** Head over to our site and browse our models. It’s like a candy store, but for algorithms.
2. **Experiment:** Dive into our "Spaces" and play with the latest AI demos.
3. **Build:** Use our tools to launch your own project.
4. **HuggingChat:** Talk to an AI that doesn’t judge your typos. 

**Hugging Face: The AI community building the future.** 
*(Warning: May cause spontaneous innovation and a sudden urge to train your own neural network.)*

**Join the conversation at huggingface.co—because the future is too important to leave to the robots alone.**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>